# Step 11: Semantic Memory and RAG (Retrieval-Augmented Generation)

This notebook demonstrates how to store facts in memory, search them semantically, and use them to improve LLM responses using RAG.

In [1]:
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import (
    AzureTextEmbedding,
    AzureChatCompletion,
)
from semantic_kernel.memory import SemanticTextMemory, VolatileMemoryStore

from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
load_dotenv(dotenv_path, override=True)

True

## Initialize Kernel and Services

We set up the Semantic Kernel with:
1. **AzureTextEmbedding**: Converts text into embeddings (vector representations)
2. **AzureChatCompletion**: Chat-based LLM for generating responses

In [2]:
kernel = Kernel()

embedding_service = AzureTextEmbedding(service_id="embedding")
chat_service = AzureChatCompletion(service_id="chat")
kernel.add_service(embedding_service)
kernel.add_service(chat_service)

## Understanding Embeddings

Let's see how embeddings work with two similar sentences.

In [3]:
TEXTS = [
    "A dog ran joyfully through the green field, chasing after butterflies in the warm afternoon sun.",
    "A happy puppy sprinted across the grassy meadow, playfully pursuing insects under the bright sky.",
]

# Generate embeddings for similar texts
text_embedded = await embedding_service.generate_embeddings(TEXTS)
print("🔢 Embeddings generated for similar texts")
print(f"Embedding dimensions: {len(text_embedded[0])}")

🔢 Embeddings generated for similar texts
Embedding dimensions: 1536


## Set Up Semantic Memory

We use `SemanticTextMemory` with `VolatileMemoryStore` (in-memory, temporary storage).

In [4]:
memory = SemanticTextMemory(
    storage=VolatileMemoryStore(), 
    embeddings_generator=embedding_service
)

C:\Users\rshukla\AppData\Local\Temp\ipykernel_22628\1488111371.py:2: DeprecationWarning: This class will be removed in a future version. Please use the InMemoryStore and Collection instead.
  storage=VolatileMemoryStore(),
C:\Users\rshukla\AppData\Local\Temp\ipykernel_22628\1488111371.py:1: DeprecationWarning: This class will be removed in a future version.
  memory = SemanticTextMemory(


## Save Facts to Memory

Let's store some travel-related facts.

In [5]:
await memory.save_information(
    collection="travel_notes",  
    id="note1", 
    text="User is currently in Barcelona.", 
)
await memory.save_information(
    collection="travel_notes",
    id="note2",
    text="User enjoys modern art museums and seaside walks.",
)
await memory.save_information(
    collection="travel_notes",
    id="note3",
    text="Today is Saturday and the user is free in the afternoon.",
)

print("✅ Facts saved to memory!")

✅ Facts saved to memory!


## Semantic Search

Now we'll search memory for relevant facts using semantic similarity.

In [6]:
query = "What should I recommend for this afternoon?"

results = await memory.search(collection="travel_notes", query=query, limit=3)

print(f"\n🔍 Semantic Query: {query}")
for r in results:
    print(f"✅ Match: '{r.text}' (score: {r.relevance:.2f})")


🔍 Semantic Query: What should I recommend for this afternoon?
✅ Match: 'Today is Saturday and the user is free in the afternoon.' (score: 0.79)
✅ Match: 'User enjoys modern art museums and seaside walks.' (score: 0.75)
✅ Match: 'User is currently in Barcelona.' (score: 0.73)


## RAG: Response WITH Memory Context

Let's use the top match as context for the LLM.

In [7]:
context_info = results[0].text
prompt_with_context = f"Based on this context: '{context_info}', what can I suggest to do this afternoon?"
response_with_context = await kernel.invoke_prompt(prompt_with_context)

print("\n--- 🧠 LLM Response WITH Memory Context ---")
print(response_with_context)


--- 🧠 LLM Response WITH Memory Context ---
Here are a few suggestions for how you can spend this free afternoon:

1. **Outdoor Activities**:
   - Go for a walk in the park, hike a nearby trail, or spend time in nature.
   - Plan a picnic and enjoy the fresh air.

2. **Fitness & Wellness**:
   - Try yoga, meditation, or a new workout.
   - Visit a spa or dedicate the afternoon to self-care.

3. **Creative Hobbies**:
   - Paint, craft, or work on a DIY project.
   - Write in a journal, blog, or explore photography.

4. **Relaxation**:
   - Curl up with a book you've been meaning to read.
   - Watch a favorite movie or binge an interesting series.

5. **Socializing**:
   - Catch up with a friend over coffee or plan a casual hangout.
   - Attend a local event or farmers' market in your area.

6. **Explore Your Community**:
   - Visit a museum, gallery, or historical site nearby.
   - Take strolls around your town or city, exploring new spots.

7. **Try Something New**:
   - Experiment wit

## Response WITHOUT Memory Context

For comparison, let's ask the same question without context.

In [8]:
prompt_without_context = "What can I suggest to do this afternoon?"
response_without_context = await kernel.invoke_prompt(prompt_without_context)

print("\n--- ❓ LLM Response WITHOUT Memory ---")
print(response_without_context)


--- ❓ LLM Response WITHOUT Memory ---
Certainly! Here are some ideas depending on your mood and interests:

### Outdoorsy Options:
1. **Take a Nature Walk**: Visit a local park, forest, or beach and enjoy the fresh air.
2. **Go for a Picnic**: Pack some snacks and relax in the sun at a nearby green space.
3. **Explore Your City or Town**: Check out a new neighborhood, landmark, or street market.
4. **Cycle Around**: Grab your bike or rent one to explore the area.
5. **Gardening**: Plant something new or spruce up your garden.

### Active Options:
6. **Attend a Gym Class**: Try yoga, dance, or a workout session.
7. **Do a DIY Home Workout**: Find a fun exercise routine on YouTube.
8. **Try a New Sport**: Go bowling, rock climbing, or play tennis.
9. **Take a Swim**: Head to the pool or a nearby lake.

### Chill and Creative Options:
10. **Read a Book or Magazine**: Find a quiet spot and get lost in a story.
11. **Start a DIY Project**: Create art, upcycle furniture, or try some crafts.

## Summary

You've now seen Retrieval-Augmented Generation (RAG) in action:
1. ✅ Store facts in semantic memory
2. 🔍 Search using semantic similarity
3. 🧠 Use retrieved context to improve LLM responses

This makes LLMs more reliable, accurate, and contextual!